# SCI1070 — Semaine 4 : tabulaire, et ce que chaque agent a le droit de regarder

| Volet | Ce que vous y faites | Statut |
| --- | --- | --- |
| A | itération sur la valeur, trois tables, sur `FrozenLake-v1` | échauffement |
| B | Q-learning tabulaire sur le **même** environnement, sans modèle | échauffement |
| C | **la falaise, Sarsa contre Q-learning** | **évalué** |
| D | TD(0) contre Monte-Carlo, sur des valeurs vraies connues | facultatif |

Le laboratoire est **formatif** : la semaine 4 ne porte aucun jalon, le prochain est
en semaine 7. Mais le **volet C** est l'exercice évalué.

**Budget de calcul : quelques secondes par exécution, sur processeur central.** Tout
est tabulaire cette semaine, et il n'y a pas une ligne de PyTorch dans ce carnet.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "gymnasium[toy-text]>=1.0"], check=True)

import collections

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

GAMMA = 0.9
print("gymnasium", gym.__version__)

---
# Volet A : avec modèle, l'itération sur la valeur

**Objectif.** Réimplémenter l'agent de la section 4.5 à partir de ses trois tables,
et résoudre `FrozenLake-v1`.

**Les constantes sont celles de la section 4.5** : `GAMMA = 0.9`, 100 pas aléatoires
par tour, 20 épisodes de test, arrêt lorsque le retour moyen de test dépasse **0,80**.

In [ ]:
class AgentTabulaire:
    def __init__(self, env, graine=0):
        self.env = env
        ...        # À COMPLÉTER : les trois tables. Lesquelles, et clés par quoi ?
        env.action_space.seed(graine)
        self.etat, _ = env.reset(seed=graine)

    def jouer_n_pas_aleatoires(self, n=100):
        """Remplit les tables de récompenses et de transitions. Aucune valeur ici."""
        for _ in range(n):
            action = self.env.action_space.sample()
            nouvel_etat, recompense, termine, tronque, _ = self.env.step(action)
            ...        # À COMPLÉTER : nourrir les deux tables, puis avancer

    def valeur_action(self, etat, action):
        compteurs = self.transitions[(etat, action)]
        total = sum(compteurs.values())
        if total == 0:
            return 0.0
        return sum(
            ...        # À COMPLÉTER : une ligne, et elle contient c_i / total
            for cible, compte in compteurs.items()
        )

    def meilleure_action(self, etat):
        return max(range(self.env.action_space.n),
                   key=lambda a: self.valeur_action(etat, a))

    def balayage(self):
        for etat in range(self.env.observation_space.n):
            self.valeurs[etat] = ...        # À COMPLÉTER : où passe le max du balayage ?

### Les trois points de contrôle

Chacun se vérifie en une ligne, et chacun attrape une erreur réelle.

In [ ]:
env = gym.make("FrozenLake-v1")
agent = AgentTabulaire(env)
agent.jouer_n_pas_aleatoires(200)

# 1. Les probabilités estimées d'un couple (état, action) somment bien à 1.
cpt = agent.transitions[(0, 1)]
total = sum(cpt.values())
assert total == 0 or abs(sum(c / total for c in cpt.values()) - 1.0) < 1e-9

# 2. Un seul balayage par tour, et pas une convergence. Comptez les appels :
#    exactement un par tour.

# 3. La valeur d'un état jamais visité reste nulle, et ne lève aucune erreur.
assert agent.valeur_action(15, 0) == 0.0 or agent.transitions[(15, 0)]
print("trois points de contrôle passés")

In [ ]:
def tester(agent, env_test, episodes=20):
    """20 épisodes avec la politique GLOUTONNE. Rend le retour moyen."""
    total = 0.0
    for _ in range(episodes):
        etat, _ = env_test.reset()
        while True:
            etat, recompense, termine, tronque, _ = env_test.step(agent.meilleure_action(etat))
            total += float(recompense)
            if termine or tronque:
                break
    return total / episodes


env_test = gym.make("FrozenLake-v1")
env_test.reset(seed=0)
agent = AgentTabulaire(gym.make("FrozenLake-v1"), graine=0)

transitions_consommees = 0
for tour in range(1, 401):
    agent.jouer_n_pas_aleatoires(100)
    transitions_consommees += 100
    agent.balayage()

    score = tester(agent, env_test)
    if tour % 25 == 0:
        print(f"tour {tour:3d} : retour de test {score:.3f}")
    if score > 0.80:
        print(f"Résolu ! en {tour} tours, {transitions_consommees} transitions")
        break

In [ ]:
# La table des valeurs à la fin : seize nombres, à lire comme une grille 4 x 4.
print(np.round(np.array([agent.valeurs[s] for s in range(16)]).reshape(4, 4), 3))
print()
print("qualité réelle de la politique, sur 400 épisodes :",
      round(tester(agent, env_test, episodes=400), 3))

> **Pourquoi la boucle monte jusqu'à 400 tours.** La dispersion que la page vous
> annonce est large : sur dix graines, le seuil tombe entre le tour **37 et le tour
> 282, médiane 120**. Un plafond à cent vous ferait conclure à un bogue une fois sur
> deux, et vous chercheriez longtemps.
>
> La cause est le critère lui-même, et elle vaut d'être comprise : la politique
> optimale de `FrozenLake-v1` **glissant** réussit environ **trois fois sur quatre**.
> Dépasser 0,80 sur **vingt** épisodes demande donc un lot de test chanceux, pas une
> meilleure politique, et le nombre de tours mesure surtout l'attente de cette
> chance. La qualité réelle, mesurée sur 400 épisodes, se tient entre 0,50 et 0,74.
> C'est, par anticipation, la leçon de la semaine 7 : **une mesure sur vingt
> épisodes ne permet pas de décider**.

**Attentes du volet A.**

* Les premiers tours affichent un retour de test de **0,000**. Ce n'est pas un
  bogue : tant qu'aucun épisode n'a atteint l'objectif, il n'existe aucune
  récompense non nulle dans la table, donc rien à apprendre. Dites-le dans votre
  carnet en une phrase, avec le mot **« exploration »** dedans.
* Les seize nombres doivent dessiner une **pente croissante** du départ vers
  l'objectif, et les trous doivent valoir zéro.
* **Ajoutez une ligne de commentaire sur la ligne où votre code lit la dynamique de
  l'environnement.** Ce commentaire est le vrai livrable de ce volet. C'est la ligne
  `(compte / total) * (...)` : la fréquence empirique y tient lieu de
  \(p(s' \mid s,a)\).

---
# Volet B : sans modèle, Q-learning tabulaire

**Objectif.** Résoudre **exactement le même** environnement avec un agent qui ne
construit **aucune** table de transitions, et comparer.

**Deux réglages diffèrent de la falaise** : `gamma = 0.9` et un `alpha` plus petit,
de l'ordre de **0,1**, parce que le glissement rend chaque transition très bruitée.

In [ ]:
def politique_gloutonne(Q, etat, n_actions):
    return max(range(n_actions), key=lambda a: Q[(etat, a)])


def q_learning(env, episodes, alpha=0.1, gamma=0.9,
               epsilon_debut=1.0, epsilon_fin=0.05, graine=0):
    """Rend (Q, retours_par_episode, transitions_consommees)."""
    rng = np.random.default_rng(graine)
    env.reset(seed=graine)
    env.action_space.seed(graine)
    n_actions = env.action_space.n
    Q = collections.defaultdict(float)
    retours, consommees = [], 0

    ...        # À COMPLÉTER : la boucle de la section 4.7. La cible prend un max sur les actions de l'état suivant, et cette action n'est PAS celle que l'agent va jouer.

    return Q, np.asarray(retours, dtype=float), consommees


Q_b, retours_b, transitions_b = q_learning(gym.make("FrozenLake-v1"), episodes=40_000)
print(f"{transitions_b} transitions consommées")

In [ ]:
...        # À COMPLÉTER : le test glouton, et le nombre de transitions pour atteindre 0,80

**Attentes du volet B.**

* Q-learning y arrive aussi, et il lui faut **beaucoup plus de transitions** que le
  volet A. C'est le résultat attendu et il ne faut pas le lire comme un échec.
* La comparaison que je vous demande d'écrire n'est **pas** « lequel gagne ». Elle
  est : **qu'est-ce que chacun des deux a le droit de regarder ?** L'agent A lit une
  dynamique qu'il a fabriquée ; l'agent B ne lit qu'une suite de transitions ;
  conséquence : A peut évaluer une action qu'il n'a jamais jouée depuis cet état, B
  ne le peut pas.
* Comptez les **transitions consommées**, pas les tours ni le temps de calcul.

---
# Volet C, évalué : la falaise, Sarsa contre Q-learning

**Objectif.** Rendre **visible** la différence entre sur politique et hors politique.

**Écrivez l'environnement vous-même.** Il existe sous le nom `CliffWalking-v1` dans
la famille des environnements en texte de Gymnasium, mais une grille de 4 × 12 tient
en vingt lignes de Python et vous apprendrez davantage à l'écrire qu'à l'importer.

In [ ]:
class Falaise:
    """Grille 4 x 12. Départ en bas à gauche, objectif en bas à droite,
    falaise sur toute la rangée du bas entre les deux."""

    HAUT, BAS, GAUCHE, DROITE = 0, 1, 2, 3
    N_ACTIONS = 4

    def __init__(self, n_lignes=4, n_colonnes=12):
        self.n_lignes, self.n_colonnes = n_lignes, n_colonnes
        self.depart = (n_lignes - 1, 0)
        self.objectif = (n_lignes - 1, n_colonnes - 1)

    def index(self, case):
        ligne, colonne = case
        return ligne * self.n_colonnes + colonne

    def est_falaise(self, case):
        ligne, colonne = case
        return ligne == self.n_lignes - 1 and 0 < colonne < self.n_colonnes - 1

    def reset(self):
        self.case = self.depart
        return self.index(self.case), {}

    def step(self, action):
        ...        # À COMPLÉTER : trois règles, et une seule est subtile. 1. hors grille : on reste sur place. 2. chaque transition rapporte -1. 3. entrer dans la falaise rapporte -100 et RENVOIE AU DÉPART, sans terminer. Seul l'objectif termine.

**Protocole.** `alpha = 0.5`, `gamma = 1.0` (non actualisé), `epsilon = 0.1`,
**500 épisodes**, et les deux méthodes sur le même environnement.

Écrivez **une seule** fonction pour les deux méthodes : elles ne diffèrent que par
la composition de la cible, et les écrire séparément est la façon la plus sûre de
laisser un `max` au mauvais endroit.

In [ ]:
def controle_td(env, methode, episodes=500, alpha=0.5, gamma=1.0,
                epsilon=0.1, graine=0, max_pas=10_000):
    """`methode` vaut "sarsa" ou "q-learning". Rend (Q, retours_par_episode)."""
    assert methode in ("sarsa", "q-learning")
    rng = np.random.default_rng(graine)
    n_actions = env.N_ACTIONS
    Q = collections.defaultdict(float)
    retours = []

    def choisir(etat):
        if rng.random() < epsilon:
            return int(rng.integers(n_actions))
        return politique_gloutonne(Q, etat, n_actions)

    ...        # À COMPLÉTER : une boucle, deux cibles. Sarsa a besoin de A_{t+1}, l'action RÉELLEMENT jouée au pas suivant. Q-learning prend un max, et ne joue pas forcément l'action du max. C'est la SEULE ligne qui change.

    return Q, np.asarray(retours, dtype=float)


falaise = Falaise()
Q_sarsa, r_sarsa = controle_td(falaise, "sarsa")
Q_qlearn, r_qlearn = controle_td(falaise, "q-learning")
print(f"Sarsa      : {r_sarsa[-100:].mean():7.1f} sur les 100 derniers épisodes")
print(f"Q-learning : {r_qlearn[-100:].mean():7.1f}")

In [ ]:
def lisser(x, fenetre=10):
    return np.convolve(np.asarray(x, dtype=float), np.ones(fenetre) / fenetre, mode="valid")


fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(lisser(r_sarsa), label="Sarsa (sur politique)")
ax.plot(lisser(r_qlearn), label="Q-learning (hors politique)")
ax.set_xlabel("épisode"); ax.set_ylabel("somme des récompenses de l'épisode")
ax.set_ylim(-120, 0)
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("La falaise : la meilleure politique obtient le pire score en ligne")
plt.show()

In [ ]:
def dessiner_politique(Q, env):
    """Une flèche par case, obtenue par argmax_a Q(s,a). Dessin en caractères."""
    fleches = {env.HAUT: "^", env.BAS: "v", env.GAUCHE: "<", env.DROITE: ">"}
    ...        # À COMPLÉTER : dix lignes, et c'est la deuxième figure du volet


print("Sarsa :");        dessiner_politique(Q_sarsa, falaise)
print("\nQ-learning :"); dessiner_politique(Q_qlearn, falaise)

### La mesure qui tranche, et elle coûte dix lignes

Rejouez les deux politiques apprises de façon **purement gloutonne**,
\(\varepsilon = 0\), sur cent épisodes chacune, et comparez les scores. Puis
commentez : **laquelle gagne maintenant, et pourquoi le classement s'inverse-t-il ?**

In [ ]:
def jouer_glouton(Q, env, episodes=100, max_pas=1000):
    ...        # À COMPLÉTER : aucune exploration, juste argmax_a Q(s,a)


print(f"Sarsa, glouton      : {jouer_glouton(Q_sarsa, falaise):7.1f}")
print(f"Q-learning, glouton : {jouer_glouton(Q_qlearn, falaise):7.1f}")

**Attentes du volet C, et c'est ce que je corrigerai.**

* Les deux courbes se séparent nettement : **Sarsa se stabilise autour de −25,
  Q-learning autour de −50.** Si les vôtres se superposent, vérifiez d'abord votre
  `cible_future`.
* Les deux chemins gloutons sont **différents** : Q-learning longe le bord, Sarsa
  prend par le haut.
* **La réponse écrite, une demi-page, et c'est le cœur de l'exercice.** Votre
  raisonnement doit faire apparaître la **composition de la cible** de chaque
  méthode, la distinction entre **politique apprise et politique jouée**, et le rôle
  de \(\varepsilon\).
* Et le classement **s'inverse** en glouton : Q-learning a appris le chemin optimal,
  qui longe la falaise ; en ligne, sous \(\varepsilon\), ce chemin tombe. Sarsa a
  appris le chemin optimal **sachant qu'il explore**.

---
# Volet D, facultatif : TD(0) contre Monte-Carlo

Une marche aléatoire à cinq états, un départ au centre, deux terminaisons,
récompense 1 à droite et 0 ailleurs, sans actualisation.

In [ ]:
# Les valeurs vraies se calculent à la main, et c'est votre vérité de référence.
VALEURS_VRAIES = np.array([1 / 6, 2 / 6, 3 / 6, 4 / 6, 5 / 6])
print("valeurs vraies :", np.round(VALEURS_VRAIES, 4))


def marche_aleatoire(rng):
    """Rend la liste des états visités et la récompense finale. Cinq états : 0..4."""
    etat = 2
    visites = [etat]
    while True:
        etat += 1 if rng.random() < 0.5 else -1
        if etat < 0:
            return visites, 0.0
        if etat > 4:
            return visites, 1.0
        visites.append(etat)


rng = np.random.default_rng(0)
visites, r = marche_aleatoire(rng)
print("un épisode :", visites, "récompense finale", r)

In [ ]:
...        # À COMPLÉTER : TD(0) et Monte-Carlo à alpha constant, puis la courbe d'erreur quadratique moyenne pour TROIS valeurs d'alpha par méthode

**Critères de réussite du volet D.**

* Une courbe d'**erreur quadratique moyenne** par rapport aux valeurs vraies, pour
  **trois** valeurs de \(\alpha\) par méthode.
* Une phrase sur laquelle des deux descend le plus vite, et une phrase sur ce que
  \(\alpha\) change pour chacune. TD(0) descend plus vite parce qu'il amorce sur une
  estimation voisine au lieu d'attendre la fin de l'épisode ; \(\alpha\) trop grand
  fait osciller les deux, et empêche la convergence fine.

> **Mon message pour ce laboratoire.** Vous venez d'écrire trois agents qui
> résolvent le même genre de problème, et le travail intéressant n'est aucun des
> trois : c'est de savoir dire, pour chacun, **de quelle information il se nourrit**.